In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import os


spark = SparkSession.builder \
    .appName("Week6_PySpark_Assignment") \
    .getOrCreate()

print("Spark Version:", spark.version)


os.makedirs("data", exist_ok=True)
os.makedirs("path/to/input", exist_ok=True)


csv_data = """product_id,product_name,category,price,old_name,base_price
101,Laptop,Electronics,45000.50,revenue,38000.0
102,Phone,Electronics,15000.00,revenue,12700.0
103,Chair,Furniture,3000.75,revenue,2542.0
104,Headphones,Electronics,2500.00,revenue,2118.0
105,Table,Furniture,5000.00,revenue,4237.0
106,Tablet,Electronics,20000.00,revenue,16949.0"""

with open("data/source.csv", "w") as f:
    f.write(csv_data)


orders_data = """order_id,status,amount,region,priority
1,Completed,1500,North,High
2,Pending,800,South,Low
3,Completed,2000,North,Low
4,Completed,500,East,High
5,Shipped,1200,North,High
6,Completed,3000,West,Low"""

with open("data/orders.csv", "w") as f:
    f.write(orders_data)

from pyspark.sql import Row

user_rows = [
    Row(user_id=1, name="Alice", amount=500.0),
    Row(user_id=None, name="Bob", amount=300.0),
    Row(user_id=3, name="Carol", amount=700.0),
    Row(user_id=None, name="Dave", amount=100.0),
    Row(user_id=5, name="Eve", amount=900.0),
]

df_users = spark.createDataFrame(user_rows)
df_users.write.mode("overwrite").parquet("path/to/input")

print("All sample data created successfully.")

Spark Version: 4.0.3
All sample data created successfully.


In [3]:
# Q3: Read CSV file with header and inferSchema enabled

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("data/source.csv")

df.printSchema()
df.show()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- old_name: string (nullable = true)
 |-- base_price: double (nullable = true)

+----------+------------+-----------+-------+--------+----------+
|product_id|product_name|   category|  price|old_name|base_price|
+----------+------------+-----------+-------+--------+----------+
|       101|      Laptop|Electronics|45000.5| revenue|   38000.0|
|       102|       Phone|Electronics|15000.0| revenue|   12700.0|
|       103|       Chair|  Furniture|3000.75| revenue|    2542.0|
|       104|  Headphones|Electronics| 2500.0| revenue|    2118.0|
|       105|       Table|  Furniture| 5000.0| revenue|    4237.0|
|       106|      Tablet|Electronics|20000.0| revenue|   16949.0|
+----------+------------+-----------+-------+--------+----------+



In [4]:
# Q5: Filter Electronics category and select specific columns

df_electronics = df.select("product_id", "price") \
                   .filter(df["category"] == "Electronics")

df_electronics.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|       101|45000.5|
|       102|15000.0|
|       104| 2500.0|
|       106|20000.0|
+----------+-------+



In [5]:
# Q6: Rename old_name to new_name and cast price from String to Double

from pyspark.sql.functions import col

df_revised = df \
    .withColumnRenamed("old_name", "new_name") \
    .withColumn("price", col("price").cast("double"))

df_revised.printSchema()
df_revised.show()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- new_name: string (nullable = true)
 |-- base_price: double (nullable = true)

+----------+------------+-----------+-------+--------+----------+
|product_id|product_name|   category|  price|new_name|base_price|
+----------+------------+-----------+-------+--------+----------+
|       101|      Laptop|Electronics|45000.5| revenue|   38000.0|
|       102|       Phone|Electronics|15000.0| revenue|   12700.0|
|       103|       Chair|  Furniture|3000.75| revenue|    2542.0|
|       104|  Headphones|Electronics| 2500.0| revenue|    2118.0|
|       105|       Table|  Furniture| 5000.0| revenue|    4237.0|
|       106|      Tablet|Electronics|20000.0| revenue|   16949.0|
+----------+------------+-----------+-------+--------+----------+



In [6]:
# Q8: Filter df_orders for Completed status AND amount > 1000

df_orders = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("data/orders.csv")

df_filtered = df_orders.filter(
    (df_orders["status"] == "Completed") &
    (df_orders["amount"] > 1000)
)

df_filtered.show()

+--------+---------+------+------+--------+
|order_id|   status|amount|region|priority|
+--------+---------+------+------+--------+
|       1|Completed|  1500| North|    High|
|       3|Completed|  2000| North|     Low|
|       6|Completed|  3000|  West|     Low|
+--------+---------+------+------+--------+



In [7]:
# Q10: Add final_price column with 18% tax applied on base_price

from pyspark.sql.functions import round

df_tax = df.withColumn("final_price", round(col("base_price") * 1.18, 2))

df_tax.select("product_id", "product_name", "base_price", "final_price").show()

+----------+------------+----------+-----------+
|product_id|product_name|base_price|final_price|
+----------+------------+----------+-----------+
|       101|      Laptop|   38000.0|    44840.0|
|       102|       Phone|   12700.0|    14986.0|
|       103|       Chair|    2542.0|    2999.56|
|       104|  Headphones|    2118.0|    2499.24|
|       105|       Table|    4237.0|    4999.66|
|       106|      Tablet|   16949.0|   19999.82|
+----------+------------+----------+-----------+



In [8]:
# Q12: Read Parquet, filter null user_id, save as CSV

import os

# Read parquet file
df_parquet = spark.read.parquet("path/to/input")

print("Before filtering nulls:")
df_parquet.show()

# Filter out rows where user_id is null
df_clean = df_parquet.filter(col("user_id").isNotNull())

print("After filtering nulls:")
df_clean.show()

# Save as CSV
df_clean.write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv("path/to/output")

print("File saved successfully at path/to/output")

# Verify saved files
print(os.listdir("path/to/output"))

Before filtering nulls:
+-------+-----+------+
|user_id| name|amount|
+-------+-----+------+
|      3|Carol| 700.0|
|   NULL| Dave| 100.0|
|      5|  Eve| 900.0|
|      1|Alice| 500.0|
|   NULL|  Bob| 300.0|
+-------+-----+------+

After filtering nulls:
+-------+-----+------+
|user_id| name|amount|
+-------+-----+------+
|      3|Carol| 700.0|
|      5|  Eve| 900.0|
|      1|Alice| 500.0|
+-------+-----+------+

File saved successfully at path/to/output
['.part-00001-61b51a55-44fc-4bac-a829-68438e16046d-c000.csv.crc', 'part-00001-61b51a55-44fc-4bac-a829-68438e16046d-c000.csv', 'part-00000-61b51a55-44fc-4bac-a829-68438e16046d-c000.csv', '.part-00000-61b51a55-44fc-4bac-a829-68438e16046d-c000.csv.crc', '_SUCCESS', '._SUCCESS.crc']


In [9]:
# Q12: Load Parquet, filter null user_id, save as CSV

import os

df_parquet = spark.read.parquet("path/to/input")

print("Before filtering nulls:")
df_parquet.show()

df_cleaned = df_parquet.filter(col("user_id").isNotNull())

print("After filtering nulls:")
df_cleaned.show()

df_cleaned.write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv("path/to/output")

print("File saved successfully at path/to/output")

# Verify saved file
df_verify = spark.read.option("header", "true").csv("path/to/output")
df_verify.show()

Before filtering nulls:
+-------+-----+------+
|user_id| name|amount|
+-------+-----+------+
|      3|Carol| 700.0|
|   NULL| Dave| 100.0|
|      5|  Eve| 900.0|
|      1|Alice| 500.0|
|   NULL|  Bob| 300.0|
+-------+-----+------+

After filtering nulls:
+-------+-----+------+
|user_id| name|amount|
+-------+-----+------+
|      3|Carol| 700.0|
|      5|  Eve| 900.0|
|      1|Alice| 500.0|
+-------+-----+------+

File saved successfully at path/to/output
+-------+-----+------+
|user_id| name|amount|
+-------+-----+------+
|      3|Carol| 700.0|
|      5|  Eve| 900.0|
|      1|Alice| 500.0|
+-------+-----+------+



In [10]:
# Q14: Filter rows where region is 'North' OR priority is 'High'

df_region = df_orders.filter(
    (df_orders["region"] == "North") |
    (df_orders["priority"] == "High")
)

df_region.show()

+--------+---------+------+------+--------+
|order_id|   status|amount|region|priority|
+--------+---------+------+------+--------+
|       1|Completed|  1500| North|    High|
|       3|Completed|  2000| North|     Low|
|       4|Completed|   500|  East|    High|
|       5|  Shipped|  1200| North|    High|
+--------+---------+------+------+--------+



# Q1: Roles of Driver, Cluster Manager, and Executor in Spark

"""
DRIVER:
- Brain of the Spark application
- Runs the main() function of your program
- Converts your code into a DAG (execution plan)
- Splits DAG into stages and tasks
- Sends tasks to executors
- Tracks progress of each task

CLUSTER MANAGER:
- Manages the resources (CPU, RAM) across the cluster
- Allocates resources to Spark when application starts
- Examples: YARN, Mesos, Kubernetes, Spark Standalone
- Does NOT run your Spark code — only manages machines

EXECUTOR:
- Worker processes that actually run the tasks
- Each executor runs on a separate worker node
- Stores data in memory/disk for caching
- Sends results back to Driver after task completion
- Multiple tasks can run in parallel inside one executor

FLOW:
Driver → asks Cluster Manager for resources
Cluster Manager → assigns Worker Nodes
Driver → sends tasks to Executors on those nodes
Executors → run tasks → return results to Driver
"""

print("Q1: Driver = Brain | Cluster Manager = Resource Allocator | Executor = Worker")

# Q4: CSV vs Parquet - Storage and Performance

"""
CSV (Row-based):
- Stores data row by row
- Row1: [id, name, price, category]
- Row2: [id, name, price, category]
- To read ONE column, must read ALL columns of every row
- No compression, plain text = large file size
- Slow for analytical queries on large datasets

PARQUET (Columnar):
- Stores data column by column
- All product_ids together, all prices together
- To read ONE column, only that column is read from disk
- Built-in compression = much smaller file size
- Very fast for queries that select few columns

WHY IT MATTERS FOR PERFORMANCE:
- Suppose table has 50 columns, you need only 3
- CSV: reads all 50 columns for every row = slow
- Parquet: reads only 3 columns = 47 columns skipped

- Parquet files are 5-10x smaller than CSV
- Parquet supports Predicate Pushdown (filter at file level)
- CSV has no metadata, Parquet stores schema + statistics

USE CASE:
- CSV: small data, data sharing, human readable
- Parquet: large datasets, analytics, production pipelines
"""

print("Q4: CSV = row-based, slow for analytics | Parquet = columnar, fast + compressed")

# Q7: How DAG provides fault tolerance in Spark

"""
WHAT IS LINEAGE GRAPH (DAG):
- DAG = Directed Acyclic Graph
- Spark records every transformation applied to data
- This record is called Lineage
- It is a step-by-step history of how data was created

HOW FAULT TOLERANCE WORKS:
- Spark does NOT store data copies like Hadoop by default
- Instead it remembers HOW to recreate the data

EXAMPLE:
Step1: Read CSV
Step2: Filter category = Electronics  
Step3: Select product_id, price
Step4: Add final_price column

If worker node crashes after Step3:
- Spark does NOT restart from beginning
- Spark looks at Lineage Graph
- Finds last safe point (Step2 result on another node)
- Reruns only Step3 and Step4 on a new worker
- No data loss, no manual recovery needed

KEY POINTS:
- Lineage = automatic recovery plan
- Only failed partition is recomputed, not full dataset
- This is why Spark is fault tolerant without data replication
- .cache() or .persist() can checkpoint lineage to reduce recomputation
"""

print("Q7: DAG Lineage = Spark remembers steps, recomputes only lost partitions on failure")

# Q9: Predicate Pushdown in Parquet

"""
WHAT IS PREDICATE PUSHDOWN:
- Predicate = your filter condition (WHERE clause)
- Pushdown = push that filter DOWN to the file reading level
- Instead of reading all data then filtering
- Spark filters BEFORE loading data into memory

HOW IT WORKS WITH PARQUET:
- Parquet stores metadata + statistics per column
- Example: min/max values per row group
- min_price=100, max_price=500 in a row group
- If your filter is price > 1000
- Spark skips that entire row group without reading it

WITHOUT PREDICATE PUSHDOWN (CSV):
- Read all 10 million rows into memory
- Then apply filter
- Wasted memory and time

WITH PREDICATE PUSHDOWN (Parquet):
- Check metadata first
- Skip row groups that cannot match filter
- Load only relevant data into memory
- Much faster, much less memory used

REAL IMPACT:
- On a 100GB dataset with filter
- CSV: loads 100GB into memory
- Parquet with pushdown: might load only 10GB
- 90% less data read from disk
"""

print("Q9: Predicate Pushdown = filter at file level before loading into memory = faster queries")

# Q11: Difference between Transformations and Actions

"""
TRANSFORMATIONS:
- Operations that create a new DataFrame from existing one
- Lazy: do NOT execute immediately
- Just add a step to the DAG plan
- Return a DataFrame

Examples:
1. filter()  → filters rows based on condition
2. select()  → selects specific columns
3. withColumn() → adds or modifies a column
4. groupBy()  → groups data for aggregation
5. join()    → joins two DataFrames

ACTIONS:
- Operations that trigger actual execution
- Lazy evaluation ends here
- Spark executes the full DAG plan
- Return a result (not a DataFrame)

Examples:
1. show()    → prints rows to console
2. count()   → returns number of rows as integer
3. collect() → returns all rows to Driver as list
4. write()   → saves DataFrame to disk

KEY DIFFERENCE:
Transformation → builds the plan (lazy)
Action        → executes the plan (eager)

FLOW:
df.filter()       # transformation - plan updated
  .select()       # transformation - plan updated
  .withColumn()   # transformation - plan updated
  .show()         # ACTION - now everything runs
"""

print("Q11: Transformations = lazy plan builders | Actions = execution triggers")

# Q13: Client Mode vs Cluster Mode in Spark

"""
CLIENT MODE:
- Driver runs on the machine that submitted the job
- That machine = your laptop or edge node
- Driver stays OUTSIDE the cluster
- Results and logs come directly to your terminal
- If your laptop dies, job fails
- Good for: interactive work, debugging, development

CLUSTER MODE:
- Driver runs INSIDE the cluster on a worker node
- Cluster Manager decides which node runs the Driver
- Your machine just submits the job and disconnects
- Job continues even if your laptop shuts down
- Good for: production jobs, long running pipelines

COMPARISON:
+------------------+---------------+------------------+
| Feature          | Client Mode   | Cluster Mode     |
+------------------+---------------+------------------+
| Driver Location  | Local machine | Inside cluster   |
| If client dies   | Job fails     | Job continues    |
| Logs/Output      | Local terminal| Cluster logs     |
| Use case         | Development   | Production       |
| Latency          | Higher        | Lower            |
+------------------+---------------+------------------+

EXAMPLE:
spark-submit --deploy-mode client  app.py  # Client Mode
spark-submit --deploy-mode cluster app.py  # Cluster Mode
"""

print("Q13: Client Mode = Driver on local machine | Cluster Mode = Driver inside cluster")

# Q15: Why use show(5) instead of collect() on large datasets

"""
collect():
- Pulls ALL rows from ALL executors to the Driver
- On 1TB dataset = 1TB data moves to your Driver node
- Driver has limited RAM (maybe 16GB or 32GB)
- Result: OutOfMemoryError, crash, job failure
- Even if it works, it is extremely slow
- NEVER use on large datasets

show(5):
- Fetches only 5 rows to display
- Rest of data stays on executors
- Driver memory is not overloaded
- Fast, safe, no risk of crash
- Spark is smart enough to stop after getting 5 rows

REAL SCENARIO:
Dataset: 500 million rows, 1TB size

collect():
- Tries to bring 500M rows to Driver
- Driver crashes with OutOfMemoryError
- Job fails, cluster resources wasted

show(5):
- Fetches 5 rows
- Displays in 2 seconds
- Cluster is fine
- You got what you needed

RULE OF THUMB:
- show(n)   → for exploring data during development
- collect() → only use on small/aggregated DataFrames
             where you are sure result fits in memory
- count()   → use instead of len(df.collect())
- write()   → use to save large results to disk
"""

print("Q15: show(5) = safe preview | collect() = memory bomb on large data")